In [1]:
import pandas as pd
import utils
import plotly.graph_objects as go

In [2]:
pair = "EUR_USD"
granularity = "H1"
ma_list = [8,16,32,64,128,256]

In [3]:
df = pd.read_pickle(utils.get_data_filename(pair, granularity))

In [4]:
non_cols = ['time', 'volume']
mod_cols = [x for x in df.columns if x not in non_cols]
df[mod_cols] = df[mod_cols].apply(pd.to_numeric)

In [5]:
df_ma = df[['time', 'mid_o', 'mid_h', 'mid_l', 'mid_c']].copy()

In [6]:
for ma in ma_list:  
    df_ma[f'MA_{ma}'] = df_ma.mid_c.rolling(window=ma).mean()

In [7]:
df_ma.dropna(inplace=True)

In [8]:
df_ma.head()

,time,mid_o,mid_h,mid_l,mid_c,MA_8,MA_16,MA_32,MA_64,MA_128,MA_256
255,2023-12-26T21:00:00.000000000Z,1.10424,1.10436,1.10408,1.10420,1.103584,1.102645,1.102272,1.100755,1.097862,1.091882
256,2023-12-26T22:00:00.000000000Z,1.10424,1.10424,1.10412,1.10416,1.103775,1.102756,1.102290,1.100906,1.097950,1.091988
257,2023-12-26T23:00:00.000000000Z,1.10417,1.10443,1.10414,1.10425,1.103994,1.102882,1.102387,1.101055,1.098050,1.092100
258,2023-12-27T00:00:00.000000000Z,1.10424,1.10426,1.10358,1.10368,1.104021,1.103018,1.102479,1.101206,1.098142,1.092206
259,2023-12-27T01:00:00.000000000Z,1.10365,1.10376,1.10320,1.10363,1.104036,1.103154,1.102542,1.101351,1.098227,1.092311


In [9]:
df_plot = df_ma.iloc[-300:].copy()

In [10]:
fig = go.Figure()
fig.add_trace(go.Candlestick(
    x=df_plot.time, open=df_plot.mid_o, high=df_plot.mid_h, low=df_plot.mid_l, close=df_plot.mid_c,
    line=dict(width=1), opacity=1,
    increasing_fillcolor='#24A06B',
    decreasing_fillcolor="#CC2E3C",
    increasing_line_color='#2EC886',  
    decreasing_line_color='#FF3A4C'
))
for ma in ma_list:
    col = f'MA_{ma}'
    fig.add_trace(go.Scatter(x=df_plot.time, 
        y=df_plot[col],
        line=dict(width=2),
        line_shape='spline',
        name=col
    ))
fig.update_layout(width=1200,height=600,
    margin=dict(l=10,r=10,b=10,t=10),
    font=dict(size=10,color="#e1e1e1"),
    paper_bgcolor="#1e1e1e",
    plot_bgcolor="#1e1e1e")
fig.update_xaxes(
    gridcolor="#1f292f",
    showgrid=True,fixedrange=True,rangeslider=dict(visible=False)
)
fig.update_yaxes(
    gridcolor="#1f292f",
    showgrid=True
)
fig.show()